In [1]:
import os
import torch
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device}")

Using: cpu


In [2]:
from google.colab import drive
drive.mount('/content/drive')

PASSION_ROOT = "/content/drive/MyDrive/PASSION_MICCAI_2024"
IMG_DIR      = os.path.join(PASSION_ROOT, "images")
CSV_PATH     = os.path.join(PASSION_ROOT, "label.csv")

df = pd.read_csv(CSV_PATH)
print(f"Metadata rows: {len(df)}")
print(df['conditions_PASSION'].value_counts())

MessageError: Error: credential propagation was unsuccessful

In [ ]:
PASSION_CLASSES = {
    'Fungal':  0,
    'Scabies': 1,
    'Eczema':  2,
    'Others':  3,
}

CLASS_NAMES = ['Fungal', 'Scabies', 'Eczema', 'Others']

class PASSIONDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.img_dir   = img_dir
        self.transform = transform
        self.samples   = []

        subject_lookup = {
            row['subject_id']: {
                'label':       PASSION_CLASSES[row['conditions_PASSION']],
                'fitzpatrick': row['fitzpatrick']
            }
            for _, row in df.iterrows()
        }

        for filename in os.listdir(img_dir):
            if not filename.endswith(('.jpg', '.jpeg', '.png')):
                continue
            if '(' in filename:
                continue

            subject_id = filename.rsplit('_', 1)[0]

            if subject_id not in subject_lookup:
                continue

            info = subject_lookup[subject_id]
            self.samples.append((
                os.path.join(img_dir, filename),
                info['label'],
                info['fitzpatrick']
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, fitzpatrick = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label, fitzpatrick

In [ ]:
from sklearn.model_selection import train_test_split

patient_ids = df['subject_id'].unique()
print(f"Total unique patients: {len(patient_ids)}")

# Split patients 80/10/10
train_ids, temp_ids = train_test_split(patient_ids, test_size=0.20, random_state=42)
val_ids,   test_ids = train_test_split(temp_ids,    test_size=0.50, random_state=42)

print(f"Train patients: {len(train_ids)}")
print(f"Val patients:   {len(val_ids)}")
print(f"Test patients:  {len(test_ids)}")

train_df = df[df['subject_id'].isin(train_ids)].reset_index(drop=True)
val_df   = df[df['subject_id'].isin(val_ids)].reset_index(drop=True)
test_df  = df[df['subject_id'].isin(test_ids)].reset_index(drop=True)

print(f"\nTrain rows: {len(train_df)}")
print(f"Val rows:   {len(val_df)}")
print(f"Test rows:  {len(test_df)}")

In [ ]:
from torchvision import transforms

# Train transform
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),        # resize slightly larger
    transforms.RandomCrop(224),           # then random crop to 224
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),      # add vertical flip
    transforms.RandomRotation(20),        # increase rotation
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Val/test transform — no augmentation, just resize and normalise
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


train_dataset = PASSIONDataset(train_df, IMG_DIR, transform=train_transform)
val_dataset   = PASSIONDataset(val_df,IMG_DIR, transform=val_transform)
test_dataset  = PASSIONDataset(test_df, IMG_DIR, transform=val_transform)

print(f"Train images: {len(train_dataset)}")
print(f"Val images:   {len(val_dataset)}")
print(f"Test images:  {len(test_dataset)}")

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32,shuffle=True,num_workers=2)
val_loader =   DataLoader(val_dataset, batch_size=32,shuffle=False,num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32,shuffle=False,num_workers=2)

In [ ]:
import torch.nn as nn

def build_e2_model(num_classes=4):
    weights = models.MobileNet_V3_Small_Weights.DEFAULT
    model   = models.mobilenet_v3_small(weights=weights)

    model.classifier[3] = nn.Linear(1024, num_classes)


    model.classifier[2] = nn.Dropout(p=0.4)
    return model

model_e2 = build_e2_model(num_classes=4).to(device)
print("E2 model built")
print(f"Classifier head: {model_e2.classifier[3]}")

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model_e2.parameters(),
    lr=3e-4,              # slightly higher to start faster
    weight_decay=1e-4     # L2 regularisation — penalises large weights
)

# Reduce LR by half if val loss doesn't improve for 3 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.2
)

print("Loss, optimiser, scheduler ready")

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct    = 0
    total      = 0

    for images, labels, _ in loader:   # _ ignores fitzpatrick during training
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct    = 0
    total      = 0

    with torch.no_grad():
        for images, labels, _ in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += images.size(0)

    return total_loss / total, correct / total

In [ ]:
# NUM_EPOCHS    = 50
# PATIENCE      = 7       # stop if val accuracy doesn't improve for 7 epochs
# best_val_acc  = 0.0
# best_model_path = 'e2_best_model.pth'
# no_improve    = 0       # counter

# train_losses, val_losses = [], []
# train_accs,   val_accs   = [], []

# for epoch in range(NUM_EPOCHS):
#     train_loss, train_acc = train_one_epoch(
#         model_e2, train_loader, optimizer, criterion, device
#     )
#     val_loss, val_acc = validate(
#         model_e2, val_loader, criterion, device
#     )

#     scheduler.step(val_loss)

#     train_losses.append(train_loss)
#     val_losses.append(val_loss)
#     train_accs.append(train_acc)
#     val_accs.append(val_acc)

#     if val_acc > best_val_acc:
#         best_val_acc = val_acc
#         torch.save(model_e2.state_dict(), best_model_path)
#         no_improve = 0
#         saved = "✓ saved"
#     else:
#         no_improve += 1
#         saved = f"(no improve {no_improve}/{PATIENCE})"

#     print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
#           f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
#           f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} {saved}")

#     if no_improve >= PATIENCE:
#         print(f"\nEarly stopping triggered at epoch {epoch+1}")
#         break

# print(f"\nBest val accuracy: {best_val_acc:.4f}")

E2 — MobileNetV3-Small trained on PASSION only
Best validation accuracy: 67.8%
Stopped at epoch: 19
Notes: Overfitting observed (train acc 93% vs val acc 68%)
       Consistent with limited dataset size (~3,900 training images)

In [ ]:
# # Load best model
# model_e2.load_state_dict(torch.load('e2_best_model.pth'))


# test_loss, test_acc = validate(model_e2, test_loader, criterion, device)
# print(f"E2 Test Accuracy: {test_acc:.4f}")

In [ ]:
from google.colab import files
files.upload()   # select the kaggle.json file you just downloaded

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets list

In [ ]:
!kaggle datasets download -d andrewmvd/isic-2019 --path ./isic2019 -q
!unzip -q ./isic2019/isic-2019.zip -d ./isic2019/

In [ ]:
import os
for item in os.listdir('./isic2019'):
    print(item)

In [ ]:
import pandas as pd

gt = pd.read_csv('./isic2019/ISIC_2019_Training_GroundTruth.csv')
print("=== Ground Truth Shape ===")
print(gt.shape)
print("\n=== First 5 rows ===")
print(gt.head())

meta = pd.read_csv('./isic2019/ISIC_2019_Training_Metadata.csv')
print("\n=== Metadata Columns ===")
print(meta.columns.tolist())
print("\n=== First 5 rows ===")
print(meta.head())

img_dir = './isic2019/ISIC_2019_Training_Input'
images  = os.listdir(img_dir)
print(f"\n=== Total images: {len(images)} ===")
print("Sample filenames:")
for f in images[:5]:
    print(f"  {f}")

In [ ]:
import os

img_dir = './isic2019/ISIC_2019_Training_Input'
contents = os.listdir(img_dir)
print(f"Items in folder: {len(contents)}")
print("First 5 items:")
for item in contents[:5]:
    print(f"  {item}")

for item in contents[:3]:
    full_path = os.path.join(img_dir, item)
    if os.path.isdir(full_path):
        print(f"\n  '{item}' is a folder, checking inside:")
        inner = os.listdir(full_path)
        print(f"  Contains {len(inner)} items")
        print(f"  Sample: {inner[:3]}")

In [ ]:
ISIC_IMG_DIR = './isic2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
print(f"Total images: {len(os.listdir(ISIC_IMG_DIR))}")  # should be 25333

In [ ]:
ISIC_CLASSES = {
    'MEL':  0,   # Melanoma
    'NV':   1,   # Melanocytic nevus
    'BCC':  2,   # Basal cell carcinoma
    'AK':   3,   # Actinic keratosis
    'BKL':  4,   # Benign keratosis
    'DF':   5,   # Dermatofibroma
    'VASC': 6,   # Vascular lesion
    'SCC':  7,   # Squamous cell carcinoma
    'UNK':  8,   # Unknown
}

CLASS_NAMES_ISIC = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']

In [ ]:
class ISICDataset(Dataset):
    def __init__(self, gt_df, img_dir, transform=None):
        self.img_dir   = img_dir
        self.transform = transform
        self.samples   = []

        for _, row in gt_df.iterrows():
            img_path  = os.path.join(img_dir, row['image'] + '.jpg')

            if not os.path.exists(img_path):
                continue
            label_idx = row[CLASS_NAMES_ISIC].values.argmax()




            fitzpatrick=-1
            self.samples.append((img_path, label_idx,fitzpatrick))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, fitzpatrick = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label, fitzpatrick

In [ ]:
isic_dataset = ISICDataset(gt, ISIC_IMG_DIR, transform=val_transform)

print(f"Total ISIC samples loaded: {len(isic_dataset)}")  # should be ~25331

for i in [0, 100, 1000]:
    _, label, fitz = isic_dataset[i]
    print(f"  Sample {i} — label: {CLASS_NAMES_ISIC[label]}, fitzpatrick: {fitz}")


In [ ]:
labels = [isic_dataset.samples[i][1] for i in range(len(isic_dataset))]
for idx, name in enumerate(CLASS_NAMES_ISIC):
    count = labels.count(idx)
    pct   = count / len(labels) * 100
    print(f"  {name}: {count} ({pct:.1f}%)")

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader

indices = list(range(len(isic_dataset)))
train_idx, temp_idx = train_test_split(indices, test_size=0.20, random_state=42)
val_idx,   test_idx = train_test_split(temp_idx, test_size=0.50, random_state=42)

train_isic = Subset(isic_dataset, train_idx)
val_isic   = Subset(isic_dataset, val_idx)
test_isic  = Subset(isic_dataset, test_idx)

train_isic_loader = DataLoader(train_isic, batch_size=32, shuffle=True,  num_workers=2)
val_isic_loader   = DataLoader(val_isic,   batch_size=32, shuffle=False, num_workers=2)
test_isic_loader  = DataLoader(test_isic,  batch_size=32, shuffle=False, num_workers=2)

print(f"Train: {len(train_isic)} | Val: {len(val_isic)} | Test: {len(test_isic)}")

In [ ]:
import torch.nn as nn

def build_e3_model(num_classes=9):
    weights = models.MobileNet_V3_Small_Weights.DEFAULT
    model   = models.mobilenet_v3_small(weights=weights)

    model.classifier[3] = nn.Linear(1024, num_classes)


    model.classifier[2] = nn.Dropout(p=0.4)
    return model

model_e3 = build_e3_model(num_classes=9).to(device)
print("E3 model built")
print(f"Classifier head: {model_e3.classifier[3]}")

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model_e3.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

# Reduce LR by half if val loss doesn't improve for 3 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.2
)


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct    = 0
    total      = 0

    for images, labels, _ in loader:   # _ ignores fitzpatrick during training
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct    = 0
    total      = 0

    with torch.no_grad():
        for images, labels, _ in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += images.size(0)

    return total_loss / total, correct / total

In [ ]:
NUM_EPOCHS    = 30
PATIENCE      = 5
best_val_acc  = 0.0
best_model_path = 'e3_best_model.pth'
no_improve    = 0       # counter

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model_e3, train_isic_loader, optimizer, criterion, device
    )
    val_loss, val_acc = validate(
        model_e3, val_isic_loader, criterion, device
    )

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_e3.state_dict(), best_model_path)
        no_improve = 0
        saved = "✓ saved"
    else:
        no_improve += 1
        saved = f"(no improve {no_improve}/{PATIENCE})"

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} {saved}")

    if no_improve >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

print(f"\nBest val accuracy: {best_val_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses,   label='Val Loss',   color='orange')
axes[0].set_title('E3 — Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(train_accs, label='Train Acc', color='steelblue')
axes[1].plot(val_accs,   label='Val Acc',   color='orange')
axes[1].set_title('E3 — Accuracy Curves')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('e3_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
model_e3.load_state_dict(torch.load('e3_best_model.pth'))
test_loss, test_acc = validate(model_e3, test_isic_loader, criterion, device)
print(f"E3 Test Accuracy (ISIC test set): {test_acc:.4f}")

In [ ]:
# Build a new loader for PASSION test set with 9-class model
# We can't measure accuracy (class spaces don't match)
# so we measure confidence like E1

def evaluate_confidence_by_fitzpatrick(model, loader, device):
    model.eval()
    from collections import defaultdict
    fitz_confidences = defaultdict(list)
    all_confidences  = []

    with torch.no_grad():
        for images, labels, fitzpatricks in loader:
            images = images.to(device)
            outputs = model(images)
            probs   = F.softmax(outputs, dim=1)
            confidences, _ = torch.max(probs, dim=1)

            for conf, fitz in zip(confidences.cpu(), fitzpatricks):
                fitz_confidences[fitz.item()].append(conf.item())
                all_confidences.append(conf.item())

    avg_by_fitz = {
        fitz: sum(confs)/len(confs)
        for fitz, confs in sorted(fitz_confidences.items())
    }
    overall = sum(all_confidences) / len(all_confidences)
    return overall, avg_by_fitz

# Run on PASSION test set
overall_conf_e3, fitz_confs_e3 = evaluate_confidence_by_fitzpatrick(
    model_e3, test_loader, device
)

print("E3 — Confidence on PASSION test set (ISIC-trained model):")
print(f"Overall avg confidence: {overall_conf_e3:.4f}")
for fitz, conf in fitz_confs_e3.items():
    print(f"  Type {fitz}: {conf:.4f}")

In [ ]:
import json

e3_results = {
    "experiment":        "E3",
    "model":             "MobileNetV3-Small",
    "pretrained_on":     "ImageNet",
    "trained_on":        "ISIC 2019 (25,331 images, 9 classes)",
    "evaluated_on":      "ISIC test set + PASSION test set",
    "num_classes":       9,
    "best_val_accuracy": 0.7967,
    "isic_test_accuracy": round(test_acc, 4),
    "passion_confidence": {
        "overall": round(overall_conf_e3, 4),
        "by_fitzpatrick": {str(k): round(v,4) for k,v in fitz_confs_e3.items()}
    },
    "notes": "Trained on Western dermatology data only. "
             "Evaluated on PASSION to measure generalisation to African skin."
}

with open('e3_results.json', 'w') as f:
    json.dump(e3_results, f, indent=2)

print(json.dumps(e3_results, indent=2))

In [ ]:

ISIC_TO_PASSION = {
    0: 3,   # MEL   -> Others
    1: 3,   # NV    -> Others
    2: 3,   # BCC   -> Others
    3: 3,   # AK    -> Others
    4: 3,   # BKL   -> Others
    5: 3,   # DF    -> Others
    6: 3,   # VASC  -> Others
    7: 3,   # SCC   -> Others
    8: 3,   # UNK   -> Others
}
# Note: ISIC has NO equivalent for Fungal(0), Scabies(1), Eczema(2)
# So the best this model can ever do is predict 'Others' for everything
# and only be correct for the ~12% of PASSION images labelled 'Others'

model_e3.eval()
correct = 0
total   = 0

with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        outputs = model_e3(images)
        preds   = outputs.argmax(dim=1).cpu()

        # Map all ISIC predictions to PASSION space
        mapped_preds = torch.tensor([ISIC_TO_PASSION[p.item()] for p in preds])
        correct += (mapped_preds == labels).sum().item()
        total   += labels.size(0)

e3_passion_acc = correct / total
print(f"E3 accuracy on PASSION test set: {e3_passion_acc:.4f}")
print(f"(Best possible if predicting 'Others' for everything: ~{189/(579+471+414+189):.4f})")

In [ ]:
UNIFIED_CLASSES = {
    'MEL':    0,
    'NV':     1,
    'BCC':    2,
    'AK':     3,
    'BKL':    4,
    'DF':     5,
    'VASC':   6,
    'SCC':    7,
    'Fungal':  8,
    'Scabies':  9,
    'Eczema':  10,
}

UNIFIED_CLASS_NAMES = [
    'Melanoma', 'Melanocytic Nevus', 'Basal Cell Carcinoma',
    'Actinic Keratosis', 'Benign Keratosis', 'Dermatofibroma',
    'Vascular Lesion', 'Squamous Cell Carcinoma',
    'Fungal Infection', 'Scabies', 'Eczema'
]

NUM_UNIFIED_CLASSES = 11
print(f"Unified class space: {NUM_UNIFIED_CLASSES} classes")

In [ ]:
class UnifiedSkinDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, fitzpatrick = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label, fitzpatrick

In [ ]:
# Remap ISIC train samples — skip UNK
isic_train_samples = []
for idx in train_idx:
    img_path, label_idx, fitz = isic_dataset.samples[idx]
    isic_class_code = CLASS_NAMES_ISIC[label_idx]
    if isic_class_code == 'UNK':
        continue
    unified_label = UNIFIED_CLASSES[isic_class_code]
    isic_train_samples.append((img_path, unified_label, fitz))

# Remap ISIC val samples — skip UNK
isic_val_samples = []
for idx in val_idx:
    img_path, label_idx, fitz = isic_dataset.samples[idx]
    isic_class_code = CLASS_NAMES_ISIC[label_idx]
    if isic_class_code == 'UNK':
        continue
    unified_label = UNIFIED_CLASSES[isic_class_code]
    isic_val_samples.append((img_path, unified_label, fitz))

# Remap PASSION train samples — skip Others
passion_train_samples = []
for img_path, label_idx, fitz in train_dataset.samples:
    passion_class = CLASS_NAMES[label_idx]
    if passion_class == 'Others':
        continue
    unified_label = UNIFIED_CLASSES[passion_class]
    passion_train_samples.append((img_path, unified_label, fitz))

# Remap PASSION val samples — skip Others
passion_val_samples = []
for img_path, label_idx, fitz in val_dataset.samples:
    passion_class = CLASS_NAMES[label_idx]
    if passion_class == 'Others':
        continue
    unified_label = UNIFIED_CLASSES[passion_class]
    passion_val_samples.append((img_path, unified_label, fitz))

# Combine
all_train_samples = isic_train_samples + passion_train_samples
all_val_samples   = isic_val_samples   + passion_val_samples

print(f"ISIC train:    {len(isic_train_samples)}")
print(f"PASSION train: {len(passion_train_samples)}")
print(f"Total train:   {len(all_train_samples)}")
print(f"Total val:     {len(all_val_samples)}")

In [ ]:
train_unified = UnifiedSkinDataset(all_train_samples, transform=train_transform)
val_unified   = UnifiedSkinDataset(all_val_samples,   transform=val_transform)

# PASSION test set remapped to unified space — skip Others
passion_test_samples_unified = []
for img_path, label_idx, fitz in test_dataset.samples:
    passion_class = CLASS_NAMES[label_idx]
    if passion_class == 'Others':
        continue
    unified_label = UNIFIED_CLASSES[passion_class]
    passion_test_samples_unified.append((img_path, unified_label, fitz))

test_unified = UnifiedSkinDataset(passion_test_samples_unified, transform=val_transform)

train_unified_loader = DataLoader(train_unified, batch_size=32, shuffle=True,  num_workers=2)
val_unified_loader   = DataLoader(val_unified,   batch_size=32, shuffle=False, num_workers=2)
test_unified_loader  = DataLoader(test_unified,  batch_size=32, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_unified_loader)}")
print(f"Val batches:   {len(val_unified_loader)}")
print(f"Test images:   {len(test_unified)}")

In [ ]:
def build_e4_model(num_classes=11):
    weights = models.MobileNet_V3_Small_Weights.DEFAULT
    model   = models.mobilenet_v3_small(weights=weights)
    model.classifier[3] = nn.Linear(1024, num_classes)
    model.classifier[2] = nn.Dropout(p=0.4)
    return model

model_e4 = build_e4_model(num_classes=NUM_UNIFIED_CLASSES).to(device)
print(f"E4 model built — {NUM_UNIFIED_CLASSES} classes")
print(f"Classifier: {model_e4.classifier[3]}")

In [ ]:
optimizer_e4 = optim.Adam(
    model_e4.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler_e4 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_e4, mode='min', patience=3, factor=0.5
)

In [ ]:
NUM_EPOCHS      = 30
PATIENCE        = 5
best_val_acc    = 0.0
best_model_path = 'e4_best_model.pth'
no_improve      = 0

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model_e4, train_unified_loader, optimizer_e4, criterion, device
    )
    val_loss, val_acc = validate(
        model_e4, val_unified_loader, criterion, device
    )

    scheduler_e4.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_e4.state_dict(), best_model_path)
        no_improve = 0
        saved = "✓ saved"
    else:
        no_improve += 1
        saved = f"(no improve {no_improve}/{PATIENCE})"

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} {saved}")

    if no_improve >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

print(f"\nBest val accuracy: {best_val_acc:.4f}")

In [ ]:
from collections import defaultdict

def evaluate_fitzpatrick(model, loader, device):
    model.eval()
    fitz_correct = defaultdict(int)
    fitz_total   = defaultdict(int)

    with torch.no_grad():
        for images, labels, fitzpatricks in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds   = outputs.argmax(dim=1)

            for pred, label, fitz in zip(preds.cpu(), labels.cpu(), fitzpatricks):
                fitz_total[fitz.item()]   += 1
                if pred == label:
                    fitz_correct[fitz.item()] += 1

    results = {
        fitz: fitz_correct[fitz] / fitz_total[fitz]
        for fitz in sorted(fitz_total)
        if fitz_total[fitz] > 0
    }
    return results

fitz_accs = evaluate_fitzpatrick(model_e2, test_loader, device)

print("E2 — Accuracy by Fitzpatrick type (test set):")
for fitz, acc in fitz_accs.items():
    bar = "█" * int(acc * 30)
    print(f"  Type {fitz}: {acc:.4f}  {bar}")

dark_accs = {k: v for k, v in fitz_accs.items() if k in [4, 5, 6]}
disparity = max(dark_accs.values()) - min(dark_accs.values())
print(f"\nDark skin tone accuracy (IV-VI): {dark_accs}")
print(f"Max disparity across IV-VI:      {disparity:.4f}")

In [ ]:
model_e4.load_state_dict(torch.load('e4_best_model.pth'))

# Overall test accuracy on unified PASSION test set
test_loss, test_acc = validate(model_e4, test_unified_loader, criterion, device)
print(f"E4 Test Accuracy (PASSION test set): {test_acc:.4f}")

# Fitzpatrick-stratified accuracy
fitz_accs_e4 = evaluate_fitzpatrick(model_e4, test_unified_loader, device)

print("\nE4 — Accuracy by Fitzpatrick type:")
for fitz, acc in fitz_accs_e4.items():
    bar = "█" * int(acc * 30)
    print(f"  Type {fitz}: {acc:.4f}  {bar}")

dark_accs_e4  = {k: v for k, v in fitz_accs_e4.items() if k in [4, 5, 6]}
disparity_e4  = max(dark_accs_e4.values()) - min(dark_accs_e4.values())
print(f"\nDark skin disparity (IV-VI): {disparity_e4:.4f}")

In [ ]:
import json

e4_results = {
    "experiment":         "E4",
    "model":              "MobileNetV3-Small",
    "pretrained_on":      "ImageNet",
    "trained_on":         "ISIC 2019 + PASSION combined (11 unified classes)",
    "evaluated_on":       "PASSION test set",
    "num_classes":        NUM_UNIFIED_CLASSES,
    "best_val_accuracy":  round(best_val_acc, 4),
    "test_accuracy":      round(test_acc, 4),
    "fitzpatrick_accuracies": {
        str(k): round(v, 4) for k, v in fitz_accs_e4.items()
    },
    "dark_skin_disparity": round(disparity_e4, 4),
    "notes": "Combined ISIC + PASSION with unified 11-class schema. "
             "Others (PASSION) and UNK (ISIC) dropped from training."
}

with open('e4_results.json', 'w') as f:
    json.dump(e4_results, f, indent=2)

print(json.dumps(e4_results, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses,   label='Val Loss',   color='orange')
axes[0].set_title('E4 — Loss Curves (ISIC + PASSION)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(train_accs, label='Train Acc', color='steelblue')
axes[1].plot(val_accs,   label='Val Acc',   color='orange')
axes[1].set_title('E4 — Accuracy Curves (ISIC + PASSION)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('e4_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()